The LLM model is supported by the Azure OPENAI service. One can also use OPENAI API to run this code.

In [1]:
import os
import sys
parent_dir = os.path.abspath('../..')
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from eval_utils import compare_MLE, data_profiling, cate_info, col_check,  transform_label
from openai import OpenAI
from sklearn.preprocessing import LabelBinarizer
import os
import json
import pickle
from model_glm import MultiAgentGAN

In [2]:
import os
import pandas as pd
from sklearn.datasets import fetch_openml
from pandas.api.types import is_numeric_dtype

# ========== 1. 下载 Adult 数据集 ==========
adult = fetch_openml(
    name="adult", 
    version=2, 
    as_frame=True
)

data = adult.frame

# ========== 2. 处理缺失值 ==========
# OpenML Adult 数据中缺失值用 "?" 表示
data = data.replace("?", pd.NA)

for col in data.columns:
    if is_numeric_dtype(data[col]):
        # 数值列：用中位数填充
        data[col] = data[col].fillna(data[col].median())
    else:
        # 类别列：用众数填充
        data[col] = data[col].fillna(data[col].mode()[0])

# ========== 3. 重命名标签列 ==========
# OpenML 原始标签列叫 "class"，你后续代码用的是 "Income"
data = data.rename(columns={"class": "Income"})

# ========== 4. 保存为项目需要的文件 ==========
os.makedirs("data", exist_ok=True)
save_path = "data/adult_imputed.csv"
data.to_csv(save_path, index=False)

print(f"✅ Adult dataset downloaded and saved to {save_path}")
print("Columns:", list(data.columns))
print("Shape:", data.shape)

✅ Adult dataset downloaded and saved to data/adult_imputed.csv
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'Income']
Shape: (48842, 15)


In [3]:
data = pd.read_csv('data/adult_imputed.csv')

In [4]:
# data100 = pd.read_csv("../sample/Adult/data100.csv")
# data200 = pd.read_csv("../sample/Adult/data200.csv")
# data400 = pd.read_csv("../sample/Adult/data400.csv")
# data800 = pd.read_csv("../sample/Adult/data800.csv")
# data_test = pd.read_csv("../sample/Adult/data_test.csv")

In [5]:
len(data)

48842

In [6]:
data.shape

(48842, 15)

In [7]:
# data.drop(columns=['Unnamed: 0'], inplace=True)
data = data.rename(columns = {'class': 'Income'})

In [8]:
x_cols = ['age', 'workclass', 'education', 'education-num',
          'marital-status', 'occupation', 'relationship', 'race', 'sex',
          'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
y_col = 'Income'
cate_var = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'Income']
bool_var = []
cols = x_cols + [y_col]

In [9]:
# api_version = "2023-05-15"

# api_key1 = 777  # Your api_key
# azure_endpoint1 = "xxx.com/"  # Your end point

# api_key2 = 777  # Your api_key
# azure_endpoint2 = "xxx.com/"  # Your end point

gen_client = OpenAI(
    api_key="sk-fe430666ae7444b8932bb8165c70fda8",
    base_url="https://api.deepseek.com"
)

opt_client = OpenAI(
    api_key="sk-fe430666ae7444b8932bb8165c70fda8",
    base_url="https://api.deepseek.com"
)

gen_model_nm = 'deepseek-chat'

opt_model_nm = 'deepseek-chat'

params = {
    'max_depth': 3,  # the maximum depth of each tree
    'eta': 0.3,  # the training step for each iteration
    # logistic regression for binary classification, output probability
    'objective': 'binary:logistic',
    'eval_metric': 'logloss'  # evaluation metric for binary classification
}
data_desc = "The dataset include people's social economic factors and demographics with the label that indicates whether their income is higher than 50k."
num_var = list((set(cols) - set(cate_var)) - set(bool_var))
num_var


['age', 'hours-per-week', 'capital-gain', 'education-num', 'capital-loss']

In [10]:
data = data[cols].copy()

In [11]:
data_test = data.sample(200, random_state=42)
data_curated = data.drop(data_test.index)
data100 = data_curated.sample(n = 100, replace=False, random_state=42)
data200 = data_curated.sample(n = 200, replace=False, random_state=42)
data400 = data_curated.sample(n = 400, replace=False, random_state=42)
data800 = data_curated.sample(n = 800, replace=False, random_state=42)
print(set(data100.index).intersection(set(data_test.index)))
data100.reset_index(inplace=True, drop=True)
data200.reset_index(inplace=True, drop=True)
data400.reset_index(inplace=True, drop=True)
data800.reset_index(inplace=True, drop=True)

data_test.to_csv("sample/Adult/data_test.csv", index=False)
data100.to_csv("sample/Adult/data100.csv", index=False)
data200.to_csv("sample/Adult/data200.csv", index=False)
data400.to_csv("sample/Adult/data400.csv", index=False)
data800.to_csv("sample/Adult/data800.csv", index=False)

set()


In [12]:
res = {}  
model_dict = {}
# num_samples = [100, 200, 400, 800]
# sampled_datasets = [data100, data200, data400, data800]
# epochs = [5, 4, 3, 2]

num_samples = [100]
sampled_datasets = [data100]
epochs = [1]   # 关键：只跑 1 epoch

# batch_size = 50
# seed_lst = list(range(5))
batch_size = 10
seed_lst = [0]
df_test = data_test.reset_index(drop=True).copy()

In [13]:
os.makedirs("log", exist_ok=True)

# for i in range(1, 4):
for i in range(1):
    num = num_samples[i]
    sampled_data = sampled_datasets[i]
    epoch = epochs[i]
    res[str(num)] = {}
    df_comb = pd.concat([sampled_data, df_test])
    df_comb.reset_index(inplace=True, drop = True)
    metadata = data_profiling(df_comb, cate_var, bool_var, cols)
    dummy_cols = list(pd.get_dummies(df_comb[x_cols]).columns)
    cate_desc = cate_info(df_comb, cate_var)
    df_test_dummy = pd.get_dummies(df_test[x_cols])
    df_test_dummy = col_check(df_test_dummy, dummy_cols)
    X_test = df_test_dummy.to_numpy()
    y_test = df_test[y_col].to_numpy()
    log_file = f'log/adult_{num}.txt'
    # magan = MultiAgentGAN(gen_client, opt_client, gen_model_nm, opt_model_nm, params, sampled_data, cols, y_col, num_var, metadata, cate_desc, data_desc, log_file, opt_temperature=1, real_samples_num=5)
    magan = MultiAgentGAN(gen_client, opt_client, gen_model_nm, opt_model_nm, params, sampled_data, cols, y_col, num_var, metadata, cate_desc, data_desc, log_file, opt_temperature=1, real_samples_num=1)
    magan._run(batch_size, epoch)
    magan.run_with_fixed_discriminator(0, epoch, batch_size, 1)
    model_dict[str(num)] = magan
    
    for j in range(len(seed_lst)):
        seed = seed_lst[j]
        res[str(num)][str(seed)] = {}
        mallm_syn_df = magan.gen_without_optimization(num_folds=1)
        mallm_syn_df.to_csv(f'gen/adult/{num}/df_{seed}.csv', index=False)

[bnlearn] >Warning: Computing DAG with 14 nodes can take a very long time!
[bnlearn] >Computing best DAG using [hc]
[bnlearn] >Set scoring type at [bic]
[bnlearn] >Compute structure scores for model comparison (higher is better).


INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
INFO:httpx